# Version 2: Scraping Data from PDF Report
### Report: Enhanced Instant Seach <a href = 'Data/eins.pdf'> EINS </a>

Data source: PDF Report

Output: Excel file containing the tables of data

### Import Libraries

In [2]:
import itertools
import numpy as np
import pdfplumber
import pandas as pd
import re
import joblib
import sys

sys.path.append('../../Litigation Clustering/')
sys.path.append('../Ssic Mapping/')

# from litigation_clustering import label_litigation_main_cause, label_litigation_case_status, label_litigation_party_status
# from ssic_mapping import ssic_mapping

In [5]:
# Open the PDF file using pdfplumber and read all lines into a single list
dataExtraction = []
x_tolereance = 8
with pdfplumber.open('../../Dataset/eins_data/eins_sample_1.pdf') as pdf:
    for page in range(len(pdf.pages)):
        page_section = pdf.pages[page].find_tables()
        endOfPage = False
        tableCount = 0
        while not endOfPage:
            try:
                table_content = page_section[tableCount].extract(x_tolerance = x_tolereance)
                dataExtraction.append(
                    [[ele for ele in table_content_data if ele not in [None, '']] for table_content_data in table_content]
                )
                tableCount += 1
            except IndexError:
                endOfPage = True

In [7]:
dataExtraction

[[['REQUEST CRITERIA'],
  ['(You have requested to search on the following)'],
  ['Date of Request:', '01/06/20XX'],
  ['Name of Requestor:', 'XYZ INFORMATION PTE LTD'],
  ['Requested Entity Name:', 'GREENYARD PIPE SYSTEMS PTE LTD'],
  ['Requested Entity Number:', '19940XXXXW'],
  ['File Reference Number:', 'EINSG SAMPLE']],
 [['SEARCH RECORD'],
  ['Entity Name:',
   '1) GREENYARD PIPE SYSTEMS PTE LTD\n2) GREENYARD PLASTICS INTERNATIONAL PTE LIMITED'],
  ['Entity Number:', '19940XXXXW']],
 [['ACCOUNTING AND CORPORATE REGULATORY AUTHORITY'],
  ['ACCOUNTING AND CORPORATE REGULATORY AUTHORITY'],
  ['BUSINESS PROFILE (COMPANY)'],
  [],
  [],
  ['WHILST EVERY ENDEAVOUR IS MADE TO ENSURE THAT THE INFORMATION PROVIDED IS UPDATED & CORRECT, THE AUTHORITY'],
  ['DISCLAIMS ANY LIABILITY FOR ANY DAMAGE OR LOSS THAT MAY BE CAUSED AS A RESULT OF ANY ERROR OR OMISSION.']],
 [['Details of Company'],
  ['Entity Name:', 'GREENYARD PIPE SYSTEMS PTE LTD'],
  ['Entity Number:', '19940XXXXW'],
  ['Date of 

## EINS 
- Contains 20 tables in the entire report
    > REQUEST CRITERIA
    
    > SEARCH RECORD

    > Details of Company

    > Capital Structure

    > Charge(s)

    > Auditor(s)

    > Officer(s) / Authorised Representative(s)

    > SHAREHOLDER(S)

    > Compliance Record

    > Financial Data

    > Key Ratios

    > ANALYSIS

    > LITIGATION SEARCH 

        > WRIT OF SUMMONS/ ORIGINATING CLAIM

        > WINDING UP

        > OS/OA/OP/OM

    > ADDITIONAL EXPERIAN PROPRIETARY INFORMATION

    > ACCREDITATIONS

    > FINANCIAL RELATED SEARCH COUNT

    > COMMERCIAL RELATED SEARCH COUNT

    > LEGAL RELATED SEARCH COUNT

Initialising Table Dictionary

In [3]:
tablesDict = {
   'REQUEST CRITERIA':                              [],
   'SEARCH RECORD':                                 [],
   'Details of Company':                            [],
   'Capital Structure':                             [],
   'Charge(s)':                                     [],
   'Auditor(s)':                                    [],
   'Officer(s) / Authorised Representative(s)':     [],
   'SHAREHOLDER(S)':                                [],
   'Compliance Record':                             [],
   'Financial Data':                                [],
   'Key Ratios':                                    [],
   'ANALYSIS':                                      [],
   '1) WRIT OF SUMMONS/ ORIGINATING CLAIM':         [],
   '2) WINDING UP':                                 [],
   '3) OS/OA/OP/OM':                                [],
   'ADDITIONAL EXPERIAN PROPRIETARY INFORMATION':   [],
   'ACCREDITATIONS':                                [],
   'FINANCIAL RELATED SEARCH COUNT':                [],
   'COMMERCIAL RELATED SEARCH COUNT':               [],
   'LEGAL RELATED SEARCH COUNT':                    [],
}

Filtering the Respective Tabular Data

In [4]:
compiledList = []
for data in dataExtraction:
    compiledList.extend(data)

tableList = []
currentKey = None
for rowData in compiledList[:-1]:
    if len(rowData) == 0:
        continue
    elif rowData[0] in list(tablesDict.keys()):
        currentTable = currentKey
        currentKey = rowData[0]
        if len(tableList) > 0:
            tablesDict[currentTable] = tableList
            tableList = []
    else:
        tableList.append(rowData)

# Add the last table to the tablesList
tablesDict['LEGAL RELATED SEARCH COUNT'] = tableList

Processing Table Data 

Table 1: REQUEST CRITERIA

In [5]:
requestCriteriaArr = np.array(tablesDict['REQUEST CRITERIA'][1:])
requestCriteriaHeaders = [h.replace(':', '').replace('*','') for h in requestCriteriaArr[:,0]]
requestCriteriaData = requestCriteriaArr[:,1]
REQUESTCRITERIA = pd.DataFrame([requestCriteriaData], columns=requestCriteriaHeaders)
REQUESTCRITERIA

,Date of Request,Name of Requestor,Requested Entity Name,Requested Entity Number,File Reference Number
0,01/06/20XX,XYZ INFORMATION PTE LTD,GREENYARD PIPE SYSTEMS PTE LTD,19940XXXXW,EINSG SAMPLE


In [6]:
columnNames = [
    'Date of Request',
    'Name of Requestor',
    'Requested Entity Name',
    'Requested Entity Number',
    'File Reference Number', 
]
for actual, expected in itertools.zip_longest(REQUESTCRITERIA.columns, columnNames):
    assert actual == expected, f'REQUEST CRITERIA has column {actual} but should be {expected}'

In [7]:
REQUESTCRITERIA.columns = [
    'request_date', 
    'requestor_name', 
    'requested_entity_name', 
    'requested_entity_id', 
    'request_id', 
]
REQUESTCRITERIA

,request_date,requestor_name,requested_entity_name,requested_entity_id,request_id
0,01/06/20XX,XYZ INFORMATION PTE LTD,GREENYARD PIPE SYSTEMS PTE LTD,19940XXXXW,EINSG SAMPLE


Table 2: SEARCH RECORD

In [8]:
searchRecordArr = np.array(tablesDict['SEARCH RECORD'][:2])
searchRecordHeaders = [h.replace(':', '').replace('*','') for h in searchRecordArr[:,0]]
searchRecordData = searchRecordArr[:,1]
SEARCHRECORD = pd.DataFrame([searchRecordData], columns=searchRecordHeaders)
SEARCHRECORD

,Entity Name,Entity Number
0,1) GREENYARD PIPE SYSTEMS PTE LTD\n2) GREENYAR...,19940XXXXW


In [9]:
columnNames = [
    'Entity Name', 
    'Entity Number', 
]
for actual, expected in itertools.zip_longest(SEARCHRECORD.columns, columnNames):
    assert actual == expected, f'SEARCH RECORD has column {actual} but should be {expected}'

In [10]:
SEARCHRECORD.columns = [
    'entity_name', 
    'entity_id', 
]
SEARCHRECORD

,entity_name,entity_id
0,1) GREENYARD PIPE SYSTEMS PTE LTD\n2) GREENYAR...,19940XXXXW


Table 3: DETAILS OF COMPANY

In [16]:
DetailsOfCompanyArr = np.array(tablesDict['Details of Company'])
DetailsOfCompanyHeaders = [h.replace(':', '').replace('*','') for h in DetailsOfCompanyArr[:,0]]
DetailsOfCompanyData = DetailsOfCompanyArr[:,1]
DETAILSOFCOMPANY = pd.DataFrame([DetailsOfCompanyData], columns=DetailsOfCompanyHeaders)
if 'Former Name (Effective Date)' in DETAILSOFCOMPANY.columns:
    columnIndex = DETAILSOFCOMPANY.columns.get_loc('Former Name (Effective Date)')
    DETAILSOFCOMPANY.insert(columnIndex, 'Former Name', None)
    DETAILSOFCOMPANY.insert(columnIndex + 1, 'Effective Date', None)
    DETAILSOFCOMPANY['Former Name'] = DETAILSOFCOMPANY['Former Name (Effective Date)'].str.extract('(.*) \(')
    DETAILSOFCOMPANY['Effective Date'] = DETAILSOFCOMPANY['Former Name (Effective Date)'].str.extract('\((.*)\)')
    DETAILSOFCOMPANY.drop(columns='Former Name (Effective Date)', inplace=True)
DETAILSOFCOMPANY

,Entity Name,Entity Number,Date of Registration (dd/mm/yyyy),Country/Region of Incorporation/Registration,Date of Change of Name,Former Name,Effective Date,Type of Company,Registered Office Address,Date of Change of Address,Principal Activity / Activities,Status,Status Date
0,GREENYARD PIPE SYSTEMS PTE LTD,19940XXXXW,12/05/19XX,SINGAPORE,01/10/19XX,GREENYARD PLASTICS INTERNATIONAL PTE LIMITED,12/05/1994,PRIVATE COMPANY LIMITED BY SHARES,99 CHANGI NORTH WAY\n#04-04\nSINGAPORE 490000,22/05/20XX,1) PLASTIC PRODUCTS SERVICES (27220),LIVE COMPANY,12/05/19XX


In [17]:
columnNames = [
    'Entity Name', 
    'Entity Number', 
    'Date of Registration (dd/mm/yyyy)',
    'Country/Region of Incorporation/Registration',
    'Date of Change of Name', 
    'Former Name', 
    'Effective Date',
    'Type of Company', 
    'Registered Office Address',
    'Date of Change of Address', 
    'Principal Activity / Activities',
    'Status', 
    'Status Date', 
]
for actual, expected in itertools.zip_longest(DETAILSOFCOMPANY.columns, columnNames):
    assert actual == expected, f'DETAILS OF COMPANY has column {actual} but should be {expected}'

In [18]:
DETAILSOFCOMPANY.columns = [
    'entity_name', 
    'entity_id', 
    'registration_date', 
    'country_or_region_of_incorporation_or_registration', 
    'date_of_name_change', 
    'former_name', 
    'effective_date_of_former_name', 
    'company_type', 
    'registered_office_address', 
    'date_of_address_change', 
    'principal_activities', 
    'status', 
    'status_date', 
]
DETAILSOFCOMPANY

,entity_name,entity_id,registration_date,country_or_region_of_incorporation_or_registration,date_of_name_change,former_name,effective_date_of_former_name,company_type,registered_office_address,date_of_address_change,principal_activities,status,status_date
0,GREENYARD PIPE SYSTEMS PTE LTD,19940XXXXW,12/05/19XX,SINGAPORE,01/10/19XX,GREENYARD PLASTICS INTERNATIONAL PTE LIMITED,12/05/1994,PRIVATE COMPANY LIMITED BY SHARES,99 CHANGI NORTH WAY\n#04-04\nSINGAPORE 490000,22/05/20XX,1) PLASTIC PRODUCTS SERVICES (27220),LIVE COMPANY,12/05/19XX


In [19]:
ssic_numbers = DETAILSOFCOMPANY['principal_activities'].apply(lambda x: re.findall(r'\((\d+)\)', x)[0])
mapping = pd.read_csv('../../Dataset/ssic_data/ssic_mapping.csv', dtype=str)
DETAILSOFCOMPANY = pd.concat([DETAILSOFCOMPANY, ssic_mapping(ssic_numbers, mapping)], axis=1)
DETAILSOFCOMPANY

,entity_name,entity_id,registration_date,country_or_region_of_incorporation_or_registration,date_of_name_change,former_name,effective_date_of_former_name,company_type,registered_office_address,date_of_address_change,principal_activities,status,status_date,Section,Division,Group,Class,Sub-Class
0,GREENYARD PIPE SYSTEMS PTE LTD,19940XXXXW,12/05/19XX,SINGAPORE,01/10/19XX,GREENYARD PLASTICS INTERNATIONAL PTE LIMITED,12/05/1994,PRIVATE COMPANY LIMITED BY SHARES,99 CHANGI NORTH WAY\n#04-04\nSINGAPORE 490000,22/05/20XX,1) PLASTIC PRODUCTS SERVICES (27220),LIVE COMPANY,12/05/19XX,MANUFACTURING,MANUFACTURE OF RUBBER AND PLASTIC PRODUCTS,MANUFACTURE OF PLASTIC PRODUCTS,Plastic Product Services,Plastic product services


In [21]:
capStructureArr = np.array(tablesDict['Capital Structure'])
capStructureHeaders = [h.replace(':', '').replace('*','') for h in capStructureArr[0,:]]
capStructureData = capStructureArr[1:,:]
CAPITALSTRUCTURE = pd.DataFrame(capStructureData, columns=capStructureHeaders)
CAPITALSTRUCTURE

,Capital Structure,No. of Shares,Currency,Amount
0,ISSUED ORDINARY,"9,661,441.00","SINGAPORE, DOLLARS","9,661,441.00"
1,PAID-UP ORDINARY,-,"SINGAPORE, DOLLARS","9,661,441.00"
2,PAID-UP PREFERENCE,-,"SINGAPORE, DOLLARS",0.00
3,PAID-UP OTHERS,-,"SINGAPORE, DOLLARS",0.00


In [22]:
capStructureData = {}
capStructureHeaders = tablesDict['Capital Structure'][0]
for data in tablesDict['Capital Structure'][1:]:
    for index in range(1, len(data)):
        capStructureData[f'{data[0]} {capStructureHeaders[index]}'] = [data[index]]
CAPITALSTRUCTURE = pd.DataFrame(capStructureData)
CAPITALSTRUCTURE

,ISSUED ORDINARY No. of Shares,ISSUED ORDINARY Currency,ISSUED ORDINARY Amount,PAID-UP ORDINARY No. of Shares,PAID-UP ORDINARY Currency,PAID-UP ORDINARY Amount,PAID-UP PREFERENCE No. of Shares,PAID-UP PREFERENCE Currency,PAID-UP PREFERENCE Amount,PAID-UP OTHERS No. of Shares,PAID-UP OTHERS Currency,PAID-UP OTHERS Amount
0,"9,661,441.00","SINGAPORE, DOLLARS","9,661,441.00",-,"SINGAPORE, DOLLARS","9,661,441.00",-,"SINGAPORE, DOLLARS",0.00,-,"SINGAPORE, DOLLARS",0.00


In [23]:
columnNames = [
    'ISSUED ORDINARY No. of Shares', 
    'ISSUED ORDINARY Currency',
    'ISSUED ORDINARY Amount', 
    'PAID-UP ORDINARY No. of Shares',
    'PAID-UP ORDINARY Currency', 
    'PAID-UP ORDINARY Amount',
    'PAID-UP PREFERENCE No. of Shares', 
    'PAID-UP PREFERENCE Currency',
    'PAID-UP PREFERENCE Amount', 
    'PAID-UP OTHERS No. of Shares',
    'PAID-UP OTHERS Currency', 
    'PAID-UP OTHERS Amount', 
]
for actual, expected in itertools.zip_longest(CAPITALSTRUCTURE.columns, columnNames):
    assert actual == expected, f'CAPITAL STRUCTURE has column {actual} but should be {expected}'

In [24]:
CAPITALSTRUCTURE.columns = [
    'issued_ordinary_no_of_shares', 
    'currency_of_issued_ordinary', 
    'amount_of_issued_ordinary', 
    'paid_up_ordinary_no_of_shares',  
    'currency_of_paid_up_ordinary', 
    'amount_of_paid_up_ordinary', 
    'paid_up_preference_no_of_shares', 
    'currency_of_paid_up_preference', 
    'amount_of_paid_up_preference', 
    'paid_up_others_no_of_shares', 
    'currency_of_paid_up_others', 
    'amount_of_paid_up_others', 
]
CAPITALSTRUCTURE

,issued_ordinary_no_of_shares,currency_of_issued_ordinary,amount_of_issued_ordinary,paid_up_ordinary_no_of_shares,currency_of_paid_up_ordinary,amount_of_paid_up_ordinary,paid_up_preference_no_of_shares,currency_of_paid_up_preference,amount_of_paid_up_preference,paid_up_others_no_of_shares,currency_of_paid_up_others,amount_of_paid_up_others
0,"9,661,441.00","SINGAPORE, DOLLARS","9,661,441.00",-,"SINGAPORE, DOLLARS","9,661,441.00",-,"SINGAPORE, DOLLARS",0.00,-,"SINGAPORE, DOLLARS",0.00


Table 5: CHARGES

In [25]:
chargesData = {}
for data in tablesDict['Charge(s)']:
    for index, value in enumerate(data):
        if index % 2:
            if columnName in chargesData:
                chargesData[columnName].append(value)
            else:
                chargesData[columnName] = [value]
        else:
            columnName = value
CHARGES = pd.DataFrame.from_dict(chargesData)
CHARGES.columns = [h.replace(':', '').replace('*','') for h in CHARGES.columns]
CHARGES

,Charge No,Date of Charge,Chargee(s),Amount Secured and Currency,Status
0,C2007XXXXX,16/01/20XX,BANK NAME,"400,000,000.00 SINGAPORE, DOLLARS",CURRENT
1,C2008XXXXX,10/02/20XX,BANK NAME,ALL MONIES,PARTIAL DISCHARGE AND REMAIN IN REGISTER
2,C2013XXXXX,16/01/20XX,BANK NAME,"97,000,000.00 SINGAPORE, DOLLARS",PARTIAL SATISFACTION AND REMAIN IN REGISTER


In [26]:
columnNames = [
    'Charge No', 
    'Date of Charge', 
    'Chargee(s)',
    'Amount Secured and Currency', 
    'Status', 
]
for actual, expected in itertools.zip_longest(CHARGES.columns, columnNames):
    assert actual == expected, f'CHARGES has column {actual} but should be {expected}'

In [27]:
CHARGES.columns = [
    'charge_id', 
    'date_of_charge', 
    'chargee', 
    'amount_secured_and_currency', 
    'status', 
]
CHARGES

,charge_id,date_of_charge,chargee,amount_secured_and_currency,status
0,C2007XXXXX,16/01/20XX,BANK NAME,"400,000,000.00 SINGAPORE, DOLLARS",CURRENT
1,C2008XXXXX,10/02/20XX,BANK NAME,ALL MONIES,PARTIAL DISCHARGE AND REMAIN IN REGISTER
2,C2013XXXXX,16/01/20XX,BANK NAME,"97,000,000.00 SINGAPORE, DOLLARS",PARTIAL SATISFACTION AND REMAIN IN REGISTER


Table 6: AUDITORS

In [21]:
auditorsArr = np.array(tablesDict['Auditor(s)'])
auditorsHeader = auditorsArr[0,:]
auditorsData = auditorsArr[1:,:]
AUDITORS = pd.DataFrame(data=auditorsData, columns=auditorsHeader)
AUDITORS

,Name,Date of Appointment
0,KPMG LLP,28/06/20XX


In [22]:
columnNames = [
    'Name', 
    'Date of Appointment', 
]
for actual, expected in itertools.zip_longest(AUDITORS.columns, columnNames):
    assert actual == expected, f'AUDITORS has column {actual} but should be {expected}'

In [23]:
AUDITORS.columns = [
    'auditor_name', 
    'date_of_appointment', 
]
AUDITORS

,auditor_name,date_of_appointment
0,KPMG LLP,28/06/20XX


Table 7: OFFICERS / AUTHORISED REPRESENTATIVES

In [24]:
officersAuthRepArr = np.array(tablesDict['Officer(s) / Authorised Representative(s)'])
officersAuthRepHeaders = [h.replace(':', '').replace('*', '') for h in officersAuthRepArr[0,:]]
officersAuthRepData = officersAuthRepArr[1:,:]
OFFICERSAUTHREPRESENTATIVE = pd.DataFrame(officersAuthRepData, columns=officersAuthRepHeaders)
for columnName in ('Name\nID', 'Address\nDate of Change of Address', 'Date of Appointment/\nPosition Held'):
    if columnName in OFFICERSAUTHREPRESENTATIVE.columns:
        columnIndex = OFFICERSAUTHREPRESENTATIVE.columns.get_loc(columnName)
        splitNames = re.split('(/ \n| \n|/\n|\n)', columnName, maxsplit=1)[::2]
        for index, splitName in enumerate(splitNames):
            OFFICERSAUTHREPRESENTATIVE.insert(columnIndex + index, splitName, None)
        OFFICERSAUTHREPRESENTATIVE[splitNames] = OFFICERSAUTHREPRESENTATIVE[columnName].str.rsplit('\n', n=1, expand=True)
        OFFICERSAUTHREPRESENTATIVE.drop(columns=columnName, inplace=True)
OFFICERSAUTHREPRESENTATIVE

,Name,ID,Address,Date of Change of Address,Nationality/Citizenship,Date of Appointment,Position Held
0,GEORGIO SOUZ,XXX600M,"VIA FIASELLA 09/29 X-16000\nGENOVA,ITALY",-,ITALIAN,21/05/20XX,DIRECTOR
1,CHAN MEI MEI,GXXX1999U,"5X TAMPINES AVENUE 1\n#0X-0X TROPICA, THE\nSIN...",-,MALAYSIAN,20/02/20XX,SECRETARY
2,SHARVIL BIN JUMAAT,SXXX7999B,"1XX MILTONIA CLOSE\nSHAUGHNESSY, THE\nSINGAPOR...",19/10/20XX,SINGAPORE CITIZEN,21/05/20XX,DIRECTOR
3,LIN MEE TUN,SXXX6000J,4XX PASIR RIS DRIVE 6\n#0X-2XX\nSINGAPORE 510444,08/04/20XX,SINGAPORE P.R.,01/08/20XX,SECRETARY


In [25]:
columnNames = [
    'Name', 
    'ID', 
    'Address', 
    'Date of Change of Address',
    'Nationality/Citizenship', 
    'Date of Appointment', 
    'Position Held', 
]
for actual, expected in itertools.zip_longest(OFFICERSAUTHREPRESENTATIVE.columns, columnNames):
    assert actual == expected, f'OFFICERS / AUTH REPRESENTATIVE has column {actual} but should be {expected}'

In [26]:
OFFICERSAUTHREPRESENTATIVE.columns = [
    'employee_name', 
    'employee_id', 
    'employee_address', 
    'date_of_change_of_address', 
    'nationality_or_citizenship', 
    'date_of_appointment', 
    'position_held', 
]
OFFICERSAUTHREPRESENTATIVE

,employee_name,employee_id,employee_address,date_of_change_of_address,nationality_or_citizenship,date_of_appointment,position_held
0,GEORGIO SOUZ,XXX600M,"VIA FIASELLA 09/29 X-16000\nGENOVA,ITALY",-,ITALIAN,21/05/20XX,DIRECTOR
1,CHAN MEI MEI,GXXX1999U,"5X TAMPINES AVENUE 1\n#0X-0X TROPICA, THE\nSIN...",-,MALAYSIAN,20/02/20XX,SECRETARY
2,SHARVIL BIN JUMAAT,SXXX7999B,"1XX MILTONIA CLOSE\nSHAUGHNESSY, THE\nSINGAPOR...",19/10/20XX,SINGAPORE CITIZEN,21/05/20XX,DIRECTOR
3,LIN MEE TUN,SXXX6000J,4XX PASIR RIS DRIVE 6\n#0X-2XX\nSINGAPORE 510444,08/04/20XX,SINGAPORE P.R.,01/08/20XX,SECRETARY


Table 8: SHAREHOLDERS

In [27]:
shareholderData = {}
for index, data in enumerate(tablesDict['SHAREHOLDER(S)']):
    if index % 2:
        for column, value in zip(columns, data):
            if column in shareholderData:
                shareholderData[column].append(value)
            else:
                shareholderData[column] = [value]
    else:
        columns = data
SHAREHOLDERS = pd.DataFrame.from_dict(shareholderData)
SHAREHOLDERS.columns = [h.replace(':', '').replace('*','') for h in SHAREHOLDERS.columns]
for columnName in ('Name\nID', 'Address\nDate of Change of Address'):
    if columnName in SHAREHOLDERS.columns:
        columnIndex = SHAREHOLDERS.columns.get_loc(columnName)
        splitNames = re.split('(/\n|\n)', columnName, maxsplit=1)[::2]
        for index, splitName in enumerate(splitNames):
            SHAREHOLDERS.insert(columnIndex + index, splitName, None)
        SHAREHOLDERS[splitNames] = SHAREHOLDERS[columnName].str.rsplit('\n', n=1, expand=True)
        SHAREHOLDERS.drop(columns=columnName, inplace=True)
SHAREHOLDERS

,Name,ID,Nationality/Citizenship,Address,Date of Change of Address,Type,No Of Shares,Currency
0,GREENYARD HOLDINGS LIMITED,19880XXXXG,SINGAPORE,3XX PASIR RIS DRIVE 1\n#0X-1XX\nSINGAPORE 110113,-,ORDINARY,"9,000,000.00","SINGAPORE, DOLLARS"
1,GREENYARD SINGPORE PTE. LTD.,20000XXXXG,SINGAPORE,1XX PASIR RIS DRIVE 2\n#0X-1XX\nSINGAPORE 110111,-,ORDINARY,"661,441.00","SINGAPORE, DOLLARS"


In [28]:
columnNames = [
    'Name', 
    'ID', 
    'Nationality/Citizenship', 
    'Address',
    'Date of Change of Address', 
    'Type', 
    'No Of Shares', 
    'Currency', 
]
for actual, expected in itertools.zip_longest(SHAREHOLDERS.columns, columnNames):
    assert actual == expected, f'SHAREHOLDERS has column {actual} but should be {expected}'

In [29]:
SHAREHOLDERS.columns = [
    'shareholder_name', 
    'shareholder_id', 
    'nationality_or_citizenship', 
    'shareholder_address', 
    'date_of_change_of_address', 
    'type_of_shares', 
    'no_of_shares', 
    'currency', 
]
SHAREHOLDERS

,shareholder_name,shareholder_id,nationality_or_citizenship,shareholder_address,date_of_change_of_address,type_of_shares,no_of_shares,currency
0,GREENYARD HOLDINGS LIMITED,19880XXXXG,SINGAPORE,3XX PASIR RIS DRIVE 1\n#0X-1XX\nSINGAPORE 110113,-,ORDINARY,"9,000,000.00","SINGAPORE, DOLLARS"
1,GREENYARD SINGPORE PTE. LTD.,20000XXXXG,SINGAPORE,1XX PASIR RIS DRIVE 2\n#0X-1XX\nSINGAPORE 110111,-,ORDINARY,"661,441.00","SINGAPORE, DOLLARS"


Table 9: COMPLIANCE RECORD

In [30]:
complianceRecordArr = np.array(tablesDict['Compliance Record'])
complianceRecordHeaders = [h.replace(':', '').replace('*','') for h in complianceRecordArr[:,0]]
complianceRecordData = complianceRecordArr[:,1:]
COMPLIANCERECORD = pd.DataFrame(data=complianceRecordData.T, columns=complianceRecordHeaders)
COMPLIANCERECORD

,Date of Last AGM,Date of Last AR,Date of A/C Laid at Last AGM
0,01/03/20XX,28/03/20XX,31/12/20XX


In [31]:
columnNames = [
    'Date of Last AGM', 
    'Date of Last AR', 
    'Date of A/C Laid at Last AGM', 
]
for actual, expected in itertools.zip_longest(COMPLIANCERECORD.columns, columnNames):
    assert actual == expected, f'COMPLIANCE RECORD has column {actual} but should be {expected}'

In [32]:
COMPLIANCERECORD.columns = [
    'date_of_last_agm', 
    'date_of_last_ar', 
    'date_of_ac_laid_at_last_agm', 
]
COMPLIANCERECORD

,date_of_last_agm,date_of_last_ar,date_of_ac_laid_at_last_agm
0,01/03/20XX,28/03/20XX,31/12/20XX


Table 10: FINANCIAL DATA

In [33]:
financialData = {
    f"Current {tablesDict['Financial Data'][1][0]}": [tablesDict['Financial Data'][1][1]], 
    f"Previous {tablesDict['Financial Data'][1][0]}": [tablesDict['Financial Data'][1][3]], 
    tablesDict['Financial Data'][2][0]: [tablesDict['Financial Data'][2][1]], 
}
for data in tablesDict['Financial Data'][3:]:
    financialData[f'Current {data[0]}'] = [data[1]]
    financialData[f'Previous {data[0]}'] = [data[3]]
    financialData[f"{data[0]} ({tablesDict['Financial Data'][1][2]})"] = [data[2]]
FINANCIALDATA = pd.DataFrame(financialData)
FINANCIALDATA.columns = [h.replace(':', '').replace('*','') for h in FINANCIALDATA.columns]
FINANCIALDATA

,Current FYE,Previous FYE,CURRENCY,Current Revenue,Previous Revenue,Revenue (% CHANGE),Current Profit after tax,Previous Profit after tax,Profit after tax (% CHANGE),Current Shareholders' fund,Previous Shareholders' fund,Shareholders' fund (% CHANGE)
0,31/12/20XX,31/12/20XX,"SINGAPORE, DOLLARS","15,098,200,000","14,857,800,000",1.61,"441,600,000","396,800,000",11.29,"13,417,300,000","13,187,400,000",1.74


In [34]:
columnNames = [
    'Current FYE',
    'Previous FYE',
    'CURRENCY',
    'Current Revenue',
    'Previous Revenue',
    'Revenue (% CHANGE)', 
    'Current Profit after tax',
    'Previous Profit after tax', 
    'Profit after tax (% CHANGE)',
    "Current Shareholders' fund", 
    "Previous Shareholders' fund",
    "Shareholders' fund (% CHANGE)", 
]
for actual, expected in itertools.zip_longest(FINANCIALDATA.columns, columnNames):
    assert actual == expected, f'FINANCIAL DATA has column {actual} but should be {expected}'

In [35]:
FINANCIALDATA.columns = [
    'current_fye', 
    'previous_fye', 
    'currency', 
    'current_fye_revenue', 
    'previous_fye_revenue', 
    'percentage_change_revenue', 
    'current_fye_profit_after_tax', 
    'previous_fye_profit_after_tax', 
    'percentage_change_profit_after_tax', 
    'current_fye_shareholders_fund', 
    'previous_fye_shareholders_fund', 
    'percentage_change_shareholders_fund', 
]
FINANCIALDATA

,current_fye,previous_fye,currency,current_fye_revenue,previous_fye_revenue,percentage_change_revenue,current_fye_profit_after_tax,previous_fye_profit_after_tax,percentage_change_profit_after_tax,current_fye_shareholders_fund,previous_fye_shareholders_fund,percentage_change_shareholders_fund
0,31/12/20XX,31/12/20XX,"SINGAPORE, DOLLARS","15,098,200,000","14,857,800,000",1.61,"441,600,000","396,800,000",11.29,"13,417,300,000","13,187,400,000",1.74


Table 11: KEY RATIOS

In [36]:
keyRatiosArr = np.array(tablesDict['Key Ratios'])
keyRatiosHeaders = [h.replace(':', '').replace('*','') for h in keyRatiosArr[:,0]]
keyRatiosData = keyRatiosArr[:,1:]
KEYRATIOS = pd.DataFrame(data=keyRatiosData.T, columns=keyRatiosHeaders)
KEYRATIOS

,FYE,Profit Margin,Liquidity [times],Debt [times]
0,31/12/20XX,2.92%,1.16,0.71
1,31/12/20XX,2.67%,1.20,1.10


In [37]:
keyRatiosData = {
    f"Current {tablesDict['Key Ratios'][0][0]}": [tablesDict['Key Ratios'][0][1]], 
    f"Previous {tablesDict['Key Ratios'][0][0]}": [tablesDict['Key Ratios'][0][2]], 
}
for data in tablesDict['Key Ratios'][1:]:
    keyRatiosData[f'Current {data[0]}'] = [data[1]]
    keyRatiosData[f'Previous {data[0]}'] = [data[2]]
KEYRATIOS = pd.DataFrame(keyRatiosData)
KEYRATIOS.columns = [h.replace(':', '').replace('*','') for h in KEYRATIOS.columns]
KEYRATIOS

,Current FYE,Previous FYE,Current Profit Margin,Previous Profit Margin,Current Liquidity [times],Previous Liquidity [times],Current Debt [times],Previous Debt [times]
0,31/12/20XX,31/12/20XX,2.92%,2.67%,1.16,1.20,0.71,1.10


In [38]:
columnNames = [
    'Current FYE', 
    'Previous FYE', 
    'Current Profit Margin',
    'Previous Profit Margin', 
    'Current Liquidity [times]',
    'Previous Liquidity [times]', 
    'Current Debt [times]',
    'Previous Debt [times]', 
]
for actual, expected in itertools.zip_longest(KEYRATIOS.columns, columnNames):
    assert actual == expected, f'KEY RATIOS has column {actual} but should be {expected}'

In [39]:
KEYRATIOS.columns = [
    'current_fye', 
    'previous_fye', 
    'current_fye_profit_margin', 
    'previous_fye_profit_margin', 
    'current_fye_liquidity_times', 
    'previous_fye_liquidity_times', 
    'current_fye_debt_times', 
    'previous_fye_debt_times', 
]
KEYRATIOS

,current_fye,previous_fye,current_fye_profit_margin,previous_fye_profit_margin,current_fye_liquidity_times,previous_fye_liquidity_times,current_fye_debt_times,previous_fye_debt_times
0,31/12/20XX,31/12/20XX,2.92%,2.67%,1.16,1.20,0.71,1.10


Table 12: ANALYSIS

In [40]:
split = re.search(r'(.*) \((.*)\)(.*)', tablesDict['ANALYSIS'][1][1])
analysisData = {
    split.group(2).split()[0]: split.group(2).split()[1],
}
for data in tablesDict['ANALYSIS'][2:6]:
    analysisData[f'{split.group(1)} {data[0]}'] = [data[1]]
    analysisData[f"{tablesDict['ANALYSIS'][1][2]} {data[0]}"] = [data[2]]
ANALYSIS = pd.DataFrame(analysisData)
ANALYSIS.columns = [h.replace(':', '').replace('*','') for h in ANALYSIS.columns]
ANALYSIS

,FYE,COMPANY LEVEL Revenue Growth,INDUSTRY MEDIAN Revenue Growth,COMPANY LEVEL Return on Assets,INDUSTRY MEDIAN Return on Assets,COMPANY LEVEL Total Asset Turnover,INDUSTRY MEDIAN Total Asset Turnover,COMPANY LEVEL Net Profit Margin,INDUSTRY MEDIAN Net Profit Margin
0,31/12/20XX,28.14%,10.03%,1.82%,5.84%,27.77%,2.45%,2.92%,3.05%


In [41]:
columnNames = [
    'FYE', 
    'COMPANY LEVEL Revenue Growth', 
    'INDUSTRY MEDIAN Revenue Growth',
    'COMPANY LEVEL Return on Assets', 
    'INDUSTRY MEDIAN Return on Assets',
    'COMPANY LEVEL Total Asset Turnover',
    'INDUSTRY MEDIAN Total Asset Turnover',
    'COMPANY LEVEL Net Profit Margin', 
    'INDUSTRY MEDIAN Net Profit Margin'
]
for actual, expected in itertools.zip_longest(ANALYSIS.columns, columnNames):
    assert actual == expected, f'ANALYSIS has column {actual} but should be {expected}'

In [42]:
ANALYSIS.columns = [
    'fye', 
    'company_level_revenue_growth', 
    'industry_median_revenue_growth', 
    'company_level_return_on_assets', 
    'industry_median_return_on_assets', 
    'company_level_total_asset_turnover', 
    'industry_median_total_asset_turnover', 
    'company_level_net_profit_margin',
    'industry_median_net_profit_margin' 
]
ANALYSIS

,fye,company_level_revenue_growth,industry_median_revenue_growth,company_level_return_on_assets,industry_median_return_on_assets,company_level_total_asset_turnover,industry_median_total_asset_turnover,company_level_net_profit_margin,industry_median_net_profit_margin
0,31/12/20XX,28.14%,10.03%,1.82%,5.84%,27.77%,2.45%,2.92%,3.05%


Table 13: LITGATION SEARCH (WRIT OF SUMMONS / ORIGINATING CLAIM)

In [43]:
writOfSummsArr = tablesDict['1) WRIT OF SUMMONS/ ORIGINATING CLAIM'][4:-1]
writeOfSummmonsOriginatingClaimHeaders = [
   'SN', 
   'Court/Type', 
   'Case \nNo.', 
   'Year', 
   'Filing \nDate', 
   'Plaintiff / \nPetitioner / \nClaimant', 
   'Case Status / \nStatus Date/ \nRemarks', 
   'Plaintiff / \nClaimant \nStatus / \nStatus Date', 
   'Defendant \nStatus*/ \n \nStatus Date', 
   'A mount(s)', 
   'Last Updated \nin Experian \nDatabase',
   'Main Cause'
   ]
# Creating the table for WRIT OF SUMMONS / ORIGINATING CLAIM
WRITOFSUMMONSORGINATINGCLAIM = pd.DataFrame(columns=[h.replace('\n', '').replace('*','').replace('A mount(s)', 'Amount(s)') for h in writeOfSummmonsOriginatingClaimHeaders])

# Concat and append the data to the table
for rowIndex in range(len(writOfSummsArr) // 2):
   dataList = writOfSummsArr[2*rowIndex]
   dataList.extend([writOfSummsArr[2*rowIndex+1][1]])
   WRITOFSUMMONSORGINATINGCLAIM = WRITOFSUMMONSORGINATINGCLAIM.append(pd.DataFrame([dataList], 
      columns=list(WRITOFSUMMONSORGINATINGCLAIM.columns)), ignore_index=True)

WRITOFSUMMONSORGINATINGCLAIM[['Case Status', 'Case Status Date', 'Case Status Remarks']] = WRITOFSUMMONSORGINATINGCLAIM['Case Status / Status Date/ Remarks'].apply(lambda x: pd.Series(re.split('(../../....)', str(x).replace('\n', ''), 1))[:3]) #split Case Status, Date and Remarks into separate columns
WRITOFSUMMONSORGINATINGCLAIM[['Plaintiff / Claimant Status', 'Plaintiff / Claimant Status Date']] = WRITOFSUMMONSORGINATINGCLAIM['Plaintiff / Claimant Status / Status Date'].apply(lambda x: pd.Series(re.split('(../../....)', str(x).replace('\n', ''), 1))[:2]) #split Plaintiff / Claimant Status and Date into separate columns
WRITOFSUMMONSORGINATINGCLAIM[['Defendant Status', 'Defendant Status Date']] = WRITOFSUMMONSORGINATINGCLAIM['Defendant Status/  Status Date'].apply(lambda x: pd.Series(re.split('(../../....)', str(x).replace('\n', ''), 1))[:2]) #split Defendant Status and Date into separate columns
WRITOFSUMMONSORGINATINGCLAIM = WRITOFSUMMONSORGINATINGCLAIM[[
   'SN', 
   'Court/Type', 
   'Case No.', 
   'Year', 
   'Filing Date',
   'Plaintiff / Petitioner / Claimant',
   'Case Status',
   'Case Status Date', 
   'Case Status Remarks',
   'Plaintiff / Claimant Status', 
   'Plaintiff / Claimant Status Date',
   'Defendant Status', 
   'Defendant Status Date',
   'Amount(s)',
   'Last Updated in Experian Database', 
   'Main Cause', 
   ]]

WRITOFSUMMONSORGINATINGCLAIM

,SN,Court/Type,Case No.,Year,Filing Date,Plaintiff / Petitioner / Claimant,Case Status,Case Status Date,Case Status Remarks,Plaintiff / Claimant Status,Plaintiff / Claimant Status Date,Defendant Status,Defendant Status Date,Amount(s),Last Updated in Experian Database,Main Cause
0,1,SC/WOS,10,20XX,16/02/20XX,ABC PTE LTD,PENDING,20/04/20XX,,PENDING,20/04/20XX,PENDING,20/04/20XX,"1) SGD 2,000,000.00\n2) SGD 50,000.00\n3) SGD ...",16/05/20XX,CONTRACTS - GENERAL
1,2,DC/WOS,97,20XX,17/07/20XX,SCC CAPITAL\nLTD,CONCLUDED,20/03/20XX,TRANSFERREDFROM MC5401/20XX,CONCLUDED,20/03/20XX,CONCLUDED,20/03/20XX,"1) SGD 50,000.00",20/04/20XX,GOOD SOLD AND DELIVERED
2,3,MC/OC,5401,20XX,09/06/20XX,SCC CAPITAL\nLTD,TRANSFERRED,17/07/20XX,TRANSFERRED TODC 97/20XX,-,NaN,-,NaN,-,15/01/20XX,GOOD SOLD AND DELIVERED


In [44]:
columnNames = [
    'SN', 
    'Court/Type', 
    'Case No.', 
    'Year', 
    'Filing Date',
    'Plaintiff / Petitioner / Claimant', 
    'Case Status', 
    'Case Status Date',
    'Case Status Remarks', 
    'Plaintiff / Claimant Status',
    'Plaintiff / Claimant Status Date', 
    'Defendant Status',
    'Defendant Status Date', 
    'Amount(s)',
    'Last Updated in Experian Database', 
    'Main Cause', 
]
for actual, expected in itertools.zip_longest(WRITOFSUMMONSORGINATINGCLAIM.columns, columnNames):
    assert actual == expected, f'WRIT OF SUMMONS ORGINATING CLAIM has column {actual} but should be {expected}'

In [45]:
WRITOFSUMMONSORGINATINGCLAIM.columns = [
    'sn', 
    'court_type', 
    'case_no', 
    'year', 
    'date_of_filling', 
    'plaintiff', 
    'case_status', 
    'date_of_case_status', 
    'case_status_remarks', 
    'plaintiff_status', 
    'date_of_plaitiff_status', 
    'defendant_status', 
    'date_of_defendant_status', 
    'amount', 
    'last_updated', 
    'main_cause', 
]
WRITOFSUMMONSORGINATINGCLAIM

,sn,court_type,case_no,year,date_of_filling,plaintiff,case_status,date_of_case_status,case_status_remarks,plaintiff_status,date_of_plaitiff_status,defendant_status,date_of_defendant_status,amount,last_updated,main_cause
0,1,SC/WOS,10,20XX,16/02/20XX,ABC PTE LTD,PENDING,20/04/20XX,,PENDING,20/04/20XX,PENDING,20/04/20XX,"1) SGD 2,000,000.00\n2) SGD 50,000.00\n3) SGD ...",16/05/20XX,CONTRACTS - GENERAL
1,2,DC/WOS,97,20XX,17/07/20XX,SCC CAPITAL\nLTD,CONCLUDED,20/03/20XX,TRANSFERREDFROM MC5401/20XX,CONCLUDED,20/03/20XX,CONCLUDED,20/03/20XX,"1) SGD 50,000.00",20/04/20XX,GOOD SOLD AND DELIVERED
2,3,MC/OC,5401,20XX,09/06/20XX,SCC CAPITAL\nLTD,TRANSFERRED,17/07/20XX,TRANSFERRED TODC 97/20XX,-,NaN,-,NaN,-,15/01/20XX,GOOD SOLD AND DELIVERED


In [46]:
clustering_model = joblib.load('../../../Litigation Clustering/litigation_main_cause_clustering.pkl')

WRITOFSUMMONSORGINATINGCLAIM['main_cause_category'] = WRITOFSUMMONSORGINATINGCLAIM['main_cause'].apply(lambda x: label_litigation_main_cause(clustering_model, x))

clustering_model = joblib.load('../../../Litigation Clustering/litigation_case_status_clustering.pkl')

WRITOFSUMMONSORGINATINGCLAIM['case_status_category'] = WRITOFSUMMONSORGINATINGCLAIM['case_status'].apply(lambda x: label_litigation_case_status(clustering_model, x))

clustering_model = joblib.load('../../../Litigation Clustering/litigation_party_status_clustering.pkl')

WRITOFSUMMONSORGINATINGCLAIM['plaintiff_status_category'] = WRITOFSUMMONSORGINATINGCLAIM['plaintiff_status'].apply(lambda x: label_litigation_party_status(clustering_model, x))
WRITOFSUMMONSORGINATINGCLAIM['defendant_status_category'] = WRITOFSUMMONSORGINATINGCLAIM['defendant_status'].apply(lambda x: label_litigation_party_status(clustering_model, x))

WRITOFSUMMONSORGINATINGCLAIM

,sn,court_type,case_no,year,date_of_filling,plaintiff,case_status,date_of_case_status,case_status_remarks,plaintiff_status,date_of_plaitiff_status,defendant_status,date_of_defendant_status,amount,last_updated,main_cause,main_cause_category,case_status_category,plaintiff_status_category,defendant_status_category
0,1,SC/WOS,10,20XX,16/02/20XX,ABC PTE LTD,PENDING,20/04/20XX,,PENDING,20/04/20XX,PENDING,20/04/20XX,"1) SGD 2,000,000.00\n2) SGD 50,000.00\n3) SGD ...",16/05/20XX,CONTRACTS - GENERAL,Contracts,Others,Others,Others
1,2,DC/WOS,97,20XX,17/07/20XX,SCC CAPITAL\nLTD,CONCLUDED,20/03/20XX,TRANSFERREDFROM MC5401/20XX,CONCLUDED,20/03/20XX,CONCLUDED,20/03/20XX,"1) SGD 50,000.00",20/04/20XX,GOOD SOLD AND DELIVERED,Others,Concluded,Others,Others
2,3,MC/OC,5401,20XX,09/06/20XX,SCC CAPITAL\nLTD,TRANSFERRED,17/07/20XX,TRANSFERRED TODC 97/20XX,-,NaN,-,NaN,-,15/01/20XX,GOOD SOLD AND DELIVERED,Others,Others,Others,Others


Table 14: LITGATION SEARCH (WINDING UP)

In [47]:
windingUpArr = tablesDict['2) WINDING UP'][1:-1]

# Creating the table for WRIT OF SUMMONS / ORIGINATING CLAIM
WINDINGUP = pd.DataFrame(columns=[h.replace('\n', ' ').replace('*','') for h in windingUpArr[0]])

for row in windingUpArr[1:]:
   WINDINGUP = WINDINGUP.append(pd.DataFrame([row], 
      columns=list(WINDINGUP.columns)), ignore_index=True)

WINDINGUP[['Case Status', 'Case Status Date']] = WINDINGUP['Case Status / Status Date'].apply(lambda x: pd.Series(re.split('(../....)', str(x).replace('\n', ''), 1))[:2]) #split Case Status and Date into separate columns
WINDINGUP[['Other Party Status', 'Other Party Status Date']] = WINDINGUP['Other Party Status/ Status Date'].apply(lambda x: pd.Series(re.split('(../....)', str(x).replace('\n', ''), 1))[:2]) #split Other Party Status and Date into separate columns
WINDINGUP[['Defendant Status', 'Defendant Status Date']] = WINDINGUP['Defendant Status/ Status Date'].apply(lambda x: pd.Series(re.split('(../....)', str(x).replace('\n', ''), 1))[:2]) #split Defendant Status and Date into separate columns
WINDINGUP = WINDINGUP[[
   'Type',
   'Case No.',
   'Year',
   'Filing Date',
   'Other Party Name',
   'Case Status', 
   'Case Status Date',
   'Other Party Status',
   'Other Party Status Date',
   'Defendant Status',
   'Defendant Status Date',
   'Amount(s)',
   'Last Updated in Experian Database'
]]
WINDINGUP

,Type,Case No.,Year,Filing Date,Other Party Name,Case Status,Case Status Date,Other Party Status,Other Party Status Date,Defendant Status,Defendant Status Date,Amount(s),Last Updated in Experian Database
0,WINDING UP,123,20XX,02/06/20XX,ABC & CO,CONCLUDED,02/09/20XX,DISCONTINUED/WITHDRAWN,02/09/20XX,CONCLUDED,02/09/20XX,"SGD 2,000,000.00",07/11/20XX


In [48]:
columnNames = [
    'Type', 
    'Case No.', 
    'Year', 
    'Filing Date', 
    'Other Party Name',
    'Case Status', 
    'Case Status Date', 
    'Other Party Status',
    'Other Party Status Date', 
    'Defendant Status', 
    'Defendant Status Date',
    'Amount(s)', 
    'Last Updated in Experian Database', 
]
for actual, expected in itertools.zip_longest(WINDINGUP.columns, columnNames):
    assert actual == expected, f'WINDING UP has column {actual} but should be {expected}'

In [49]:
WINDINGUP.columns = [
    'type', 
    'case_no', 
    'year', 
    'date_of_filling', 
    'other_party_name', 
    'case_status', 
    'date_of_case_status', 
    'other_party_status', 
    'date_of_other_party_status', 
    'defendant_status', 
    'date_of_defendant_status', 
    'amount', 
    'last_updated'
]
WINDINGUP

,type,case_no,year,date_of_filling,other_party_name,case_status,date_of_case_status,other_party_status,date_of_other_party_status,defendant_status,date_of_defendant_status,amount,last_updated
0,WINDING UP,123,20XX,02/06/20XX,ABC & CO,CONCLUDED,02/09/20XX,DISCONTINUED/WITHDRAWN,02/09/20XX,CONCLUDED,02/09/20XX,"SGD 2,000,000.00",07/11/20XX


In [50]:
clustering_model = joblib.load('../../Litigation Clustering/litigation_case_status_clustering.pkl')

WINDINGUP['case_status_category'] = WINDINGUP['case_status'].apply(lambda x: label_litigation_case_status(clustering_model, x))

clustering_model = joblib.load('../../Litigation Clustering/litigation_party_status_clustering.pkl')

WINDINGUP['other_party_status_category'] = WINDINGUP['other_party_status'].apply(lambda x: label_litigation_party_status(clustering_model, x))
WINDINGUP['defendant_status_category'] = WINDINGUP['defendant_status'].apply(lambda x: label_litigation_party_status(clustering_model, x))

WINDINGUP

,type,case_no,year,date_of_filling,other_party_name,case_status,date_of_case_status,other_party_status,date_of_other_party_status,defendant_status,date_of_defendant_status,amount,last_updated,case_status_category,other_party_status_category,defendant_status_category
0,WINDING UP,123,20XX,02/06/20XX,ABC & CO,CONCLUDED,02/09/20XX,DISCONTINUED/WITHDRAWN,02/09/20XX,CONCLUDED,02/09/20XX,"SGD 2,000,000.00",07/11/20XX,Concluded,Others,Others


Table 15: LITIGATION SEARCH (OS/OA/OP/OM)

In [51]:
OSOAOPOMArr = tablesDict['3) OS/OA/OP/OM'][2:-1]
OSOAOPOMArr

# Initialise the table
OSOAOPOM = pd.DataFrame(columns=[h.replace('\n', ' ').replace('*','') for h in OSOAOPOMArr[0]])
for rowData in OSOAOPOMArr[1:]:
   OSOAOPOM = OSOAOPOM.append(pd.DataFrame([rowData], 
      columns=list(OSOAOPOM.columns)), ignore_index=True)

OSOAOPOM[['Case Status', 'Case Status Date']] = OSOAOPOM['Case Status / Status Date'].apply(lambda x: pd.Series(re.split('(../....)', str(x).replace('\n', ''), 1))[:2]) #split Case Status and Date into separate columns
OSOAOPOM[['Other Party Status', 'Other Party Status Date']] = OSOAOPOM['Other Party Status/ Status Date'].apply(lambda x: pd.Series(re.split('(../....)', str(x).replace('\n', ''), 1))[:2]) #split Other Party Status and Date into separate columns
OSOAOPOM[['Defendant Status', 'Defendant Status Date']] = OSOAOPOM['Defendant Status/ Status Date'].apply(lambda x: pd.Series(re.split('(../....)', str(x).replace('\n', ''), 1))[:2]) #split Defendant Status and Date into separate columns
OSOAOPOM = OSOAOPOM[[
   'Type',
   'Case No.',
   'Year',
   'Filing Date',
   'Other Party Name',
   'Case Status', 
   'Case Status Date',
   'Other Party Status',
   'Other Party Status Date',
   'Defendant Status',
   'Defendant Status Date',
   'Amount(s)',
   'Last Updated in Experian Database'
]]
OSOAOPOM

,Type,Case No.,Year,Filing Date,Other Party Name,Case Status,Case Status Date,Other Party Status,Other Party Status Date,Defendant Status,Defendant Status Date,Amount(s),Last Updated in Experian Database
0,ORIGINATING\nSUMMONS,56,20XX,15/02/20XX,XYZ & CO,CONCLUDED,11/12/20XX,CONCLUDED,11/12/20XX,CONCLUDED,11/12/20XX,"SGD 2,000,000.00",07/11/20XX


In [52]:
columnNames = [
    'Type',
    'Case No.',
    'Year',
    'Filing Date',
    'Other Party Name',
    'Case Status', 
    'Case Status Date', 
    'Other Party Status',
    'Other Party Status Date', 
    'Defendant Status', 
    'Defendant Status Date',
    'Amount(s)', 
    'Last Updated in Experian Database', 
]
for actual, expected in itertools.zip_longest(OSOAOPOM.columns, columnNames):
    assert actual == expected, f'OSOAOPOM has column {actual} but should be {expected}'

In [53]:
OSOAOPOM.columns = [
    'type', 
    'case_no', 
    'year', 
    'date_of_filing', 
    'other_party_name', 
    'case_status', 
    'date_of_case_status', 
    'other_party_status', 
    'date_of_other_party_status', 
    'defendant_status', 
    'date_of_defendant_status', 
    'amount', 
    'last_updated'
]
OSOAOPOM

,type,case_no,year,date_of_filing,other_party_name,case_status,date_of_case_status,other_party_status,date_of_other_party_status,defendant_status,date_of_defendant_status,amount,last_updated
0,ORIGINATING\nSUMMONS,56,20XX,15/02/20XX,XYZ & CO,CONCLUDED,11/12/20XX,CONCLUDED,11/12/20XX,CONCLUDED,11/12/20XX,"SGD 2,000,000.00",07/11/20XX


In [54]:
clustering_model = joblib.load('../../Litigation Clustering/litigation_case_status_clustering.pkl')

OSOAOPOM['case_status_category'] = OSOAOPOM['case_status'].apply(lambda x: label_litigation_case_status(clustering_model, x))

clustering_model = joblib.load('../../Litigation Clustering/litigation_party_status_clustering.pkl')

OSOAOPOM['other_party_status_category'] = OSOAOPOM['other_party_status'].apply(lambda x: label_litigation_party_status(clustering_model, x))
OSOAOPOM['defendant_status_category'] = OSOAOPOM['defendant_status'].apply(lambda x: label_litigation_party_status(clustering_model, x))

OSOAOPOM

,type,case_no,year,date_of_filing,other_party_name,case_status,date_of_case_status,other_party_status,date_of_other_party_status,defendant_status,date_of_defendant_status,amount,last_updated,case_status_category,other_party_status_category,defendant_status_category
0,ORIGINATING\nSUMMONS,56,20XX,15/02/20XX,XYZ & CO,CONCLUDED,11/12/20XX,CONCLUDED,11/12/20XX,CONCLUDED,11/12/20XX,"SGD 2,000,000.00",07/11/20XX,Concluded,Others,Others


Table 16: ADDITIONAL EXPERIAN PROPRIETARY INFORMATION

In [55]:
addExpPropInfoArr = np.array(tablesDict['ADDITIONAL EXPERIAN PROPRIETARY INFORMATION'])
addExpPropInfoHeaders = [h.replace(':', '').replace('*','') for h in addExpPropInfoArr[:,0]]
addExpPropInfoData = addExpPropInfoArr[:,1:]
ADDEXPPROPINFO = pd.DataFrame(addExpPropInfoData.T, columns=addExpPropInfoHeaders)
ADDEXPPROPINFO

,No. of Employees,Commercial Properties Owned
0,31834,-


In [56]:
columnNames = [
    'No. of Employees', 
    'Commercial Properties Owned', 
]
for actual, expected in itertools.zip_longest(ADDEXPPROPINFO.columns, columnNames):
    assert actual == expected, f'ADD EXP PROP INFO has column {actual} but should be {expected}'

In [57]:
ADDEXPPROPINFO.columns = [
    'num_of_employee', 
    'num_of_commerical_properties_owned', 
]
ADDEXPPROPINFO

,num_of_employee,num_of_commerical_properties_owned
0,31834,-


Table 17: ACCREDITATIONS

In [58]:
accreditationsArr = np.array(tablesDict['ACCREDITATIONS'])
accreditationsArrHeaders = [h.replace(':', '').replace('*','') for h in accreditationsArr[0]]
accreditationsArrData = accreditationsArr[1:]

ACCREDITATIONS = pd.DataFrame(accreditationsArrData, columns=accreditationsArrHeaders)
ACCREDITATIONS

,Award,Year,Ranking
0,SINGAPORE 1000 COMPANIES RANKED BY SALES/TURNOVER,20XX,XXX
1,SINGAPORE 1000 COMPANIES RANKED BY SALES/TURNOVER,20XX,XXX


In [59]:
columnNames = [
    'Award', 
    'Year', 
    'Ranking', 
]
for actual, expected in itertools.zip_longest(ACCREDITATIONS.columns, columnNames):
    assert actual == expected, f'ACCREDITATIONS has column {actual} but should be {expected}'

In [60]:
ACCREDITATIONS.columns = [
    'award_name', 
    'year_of_award', 
    'award_ranking', 
]
ACCREDITATIONS

,award_name,year_of_award,award_ranking
0,SINGAPORE 1000 COMPANIES RANKED BY SALES/TURNOVER,20XX,XXX
1,SINGAPORE 1000 COMPANIES RANKED BY SALES/TURNOVER,20XX,XXX


Table 18: FINANCIAL RELATED SEARCH COUNT

In [61]:
financialRelatedSearchCountData = {
    'year': [], 
    'month': [], 
    'purpose': [], 
    'count': [], 
}

def add(year, month, purpose, count):
    financialRelatedSearchCountData['year'].append(year)
    financialRelatedSearchCountData['month'].append(month)
    financialRelatedSearchCountData['purpose'].append(purpose)
    financialRelatedSearchCountData['count'].append(count)

for data in tablesDict['FINANCIAL RELATED SEARCH COUNT'][2:]:
    if len(data) == 14:
        year = data[0]
        add(year, 'All', 'All', data[1])
        for index in range(2, 14):
            add(year, tablesDict['FINANCIAL RELATED SEARCH COUNT'][1][index], 'All', data[index])
    else:
        for index in range(1, 13):
            add(year, tablesDict['FINANCIAL RELATED SEARCH COUNT'][1][index], data[0], data[index])
FINRELSEARCHCOUNT = pd.DataFrame(financialRelatedSearchCountData)
FINRELSEARCHCOUNT

,year,month,purpose,count
0,20XX,All,All,3
1,20XX,Jan,All,3
2,20XX,Feb,All,0
3,20XX,Mar,All,0
4,20XX,Apr,All,0
...,...,...,...,...
58,20XX,Aug,All,0
59,20XX,Sep,All,0
60,20XX,Oct,All,0
61,20XX,Nov,All,0


Table 19: COMMERCIAL RELATED SEARCH COUNT

In [62]:
commercialRelatedSearchCountData = {
    'year': [], 
    'month': [], 
    'purpose': [], 
    'count': [], 
}

def add(year, month, purpose, count):
    commercialRelatedSearchCountData['year'].append(year)
    commercialRelatedSearchCountData['month'].append(month)
    commercialRelatedSearchCountData['purpose'].append(purpose)
    commercialRelatedSearchCountData['count'].append(count)

for data in tablesDict['COMMERCIAL RELATED SEARCH COUNT'][2:]:
    if len(data) == 14:
        year = data[0]
        add(year, 'All', 'All', data[1])
        for index in range(2, 14):
            add(year, tablesDict['COMMERCIAL RELATED SEARCH COUNT'][1][index], 'All', data[index])
    else:
        for index in range(1, 13):
            add(year, tablesDict['COMMERCIAL RELATED SEARCH COUNT'][1][index], data[0], data[index])
COMRELSEARCHCOUNT = pd.DataFrame(commercialRelatedSearchCountData)
COMRELSEARCHCOUNT

,year,month,purpose,count
0,20XX,All,All,1
1,20XX,Jan,All,1
2,20XX,Feb,All,0
3,20XX,Mar,All,0
4,20XX,Apr,All,0
5,20XX,May,All,0
6,20XX,Jun,All,0
7,20XX,Jul,All,0
8,20XX,Aug,All,0
9,20XX,Sep,All,0


Table 20: LEGAL RELATED SEARCH COUNT

In [63]:
legalRelatedSearchCountData = {
    'year': [], 
    'month': [], 
    'purpose': [], 
    'count': [], 
}

def add(year, month, purpose, count):
    legalRelatedSearchCountData['year'].append(year)
    legalRelatedSearchCountData['month'].append(month)
    legalRelatedSearchCountData['purpose'].append(purpose)
    legalRelatedSearchCountData['count'].append(count)

for data in tablesDict['LEGAL RELATED SEARCH COUNT'][2:]:
    if len(data) == 14:
        year = data[0]
        add(year, 'All', 'All', data[1])
        for index in range(2, 14):
            add(year, tablesDict['LEGAL RELATED SEARCH COUNT'][1][index], 'All', data[index])
    else:
        for index in range(1, 13):
            add(year, tablesDict['LEGAL RELATED SEARCH COUNT'][1][index], data[0], data[index])
LEGRELSEARCHCOUNT = pd.DataFrame(legalRelatedSearchCountData)
LEGRELSEARCHCOUNT

,year,month,purpose,count
0,20XX,All,All,2
1,20XX,Jan,All,2
2,20XX,Feb,All,0
3,20XX,Mar,All,0
4,20XX,Apr,All,0
...,...,...,...,...
58,20XX,Aug,All,0
59,20XX,Sep,All,0
60,20XX,Oct,All,0
61,20XX,Nov,All,0


## Export all the tables to excel / csv

In [143]:
# Output Data
writer = pd.ExcelWriter(f'../Output/tables_v2.xlsx', engine='xlsxwriter')

REQUESTCRITERIA.to_excel(writer, sheet_name='Request Criteria')
SEARCHRECORD.to_excel(writer, sheet_name='Search Record')
DETAILSOFCOMPANY.to_excel(writer, sheet_name='Details of Company')
CAPITALSTRUCTURE.to_excel(writer, sheet_name='Capital Structure')
CHARGES.to_excel(writer, sheet_name='Charges')
AUDITORS.to_excel(writer, sheet_name='Auditors')
OFFICERSAUTHREPRESENTATIVE.to_excel(writer, sheet_name='Officers Auth. Rep')
SHAREHOLDERS.to_excel(writer, sheet_name='Shareholders')
COMPLIANCERECORD.to_excel(writer, sheet_name='Compliance Record')
FINANCIALDATA.to_excel(writer, sheet_name='Financial Data')
KEYRATIOS.to_excel(writer, sheet_name='Key Ratios')
ANALYSIS.to_excel(writer, sheet_name='Analysis')
WRITOFSUMMONSORGINATINGCLAIM.to_excel(writer, sheet_name='Writ of Sum. Org. Claim')
WINDINGUP.to_excel(writer, sheet_name='Winding Up')
OSOAOPOM.to_excel(writer, sheet_name='OS OA OP OM')
ADDEXPPROPINFO.to_excel(writer, sheet_name='Add Exp Prop. Info')
ACCREDITATIONS.to_excel(writer, sheet_name='Accreditations')
FINRELSEARCHCOUNT.to_excel(writer, sheet_name='Financial Related Search Count')
COMRELSEARCHCOUNT.to_excel(writer, sheet_name='Commercial Related Search Count')
LEGRELSEARCHCOUNT.to_excel(writer, sheet_name='Legal Related Search Count')

writer.close()